In [144]:
import pandas as pd
from datetime import datetime, timedelta
from functions import *
import calendar
import warnings
warnings.filterwarnings("ignore")
import shutil

date = datetime.now().date()
# date = datetime.strptime(datetime.now(), "%d-%m-%Y").date()
base_dir_format = datetime.now().strftime('%d.%m.%y')
base_dir = f's:\\{base_dir_format}'
result_directory = 's:\\playground'
create_folder_if_not_exists(result_directory)
error_count = 0
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

ftpartialname = 'NEWCNEFINTRANSRPT'
newsalespartialname = 'CNENEWCAPTURERPT.CSV'
beindate_partial_name = 'BEINDATANEWRPT'

ftFoundFiles = search_files(base_dir,ftpartialname)
newsalesFoundFiles=search_files(base_dir,newsalespartialname)
beindata_found_files = search_files(base_dir, beindate_partial_name)


print(ftFoundFiles[0])
print(newsalesFoundFiles[0])
print(date)

s:\22.01.26\29508076_NEWCNEFINTRANSRPT.CSV
s:\22.01.26\29508709_CNENEWCAPTURERPT.CSV
2026-01-22


In [145]:
def clean_text(s):
    words = s.split()
    if len(words[-1]) == 1 and len(words) >= 2:
        return " ".join(words[:-2])
    return " ".join(words[:-1])

def diff_month(d1, d2):
    return (d1.dt.year - d2.dt.year) * 12 + d1.dt.month - d2.dt.month

In [146]:
#load files
ft = pd.read_csv(ftFoundFiles[0],dtype='str',on_bad_lines='skip')
newsales = pd.read_csv(newsalesFoundFiles[0], dtype='str')
beindata = pd.read_csv(beindata_found_files[1],dtype="str")

# format dates 
newsales['Customer Created Date'] = pd.to_datetime(newsales['Customer Created Date'])
ft['Created Date'] = pd.to_datetime(ft['Created Date'])

In [147]:
print(str((date-timedelta(days=1))))

2026-01-21


In [155]:
ft_clone = ft.copy()
newsales_clone = newsales.copy()
beindata_clone = beindata.copy()

#                                                         filter files


ft_clone = ft_clone.loc[ft_clone['Doc Status']=='Posted']
ft_clone = ft_clone.loc[ft_clone['Created Date'] == str((date-timedelta(days=1)))]
ft_clone["Location"] = ft_clone['User Fullname'].apply(clean_text)
ft_clone = ft_clone.loc[ft_clone['Doc Type'].isin(['JV','Payment'])]
## link to beindata to get decoder type
merged = pd.merge(left=ft_clone,right=beindata_clone[['Decoder','Smart Card','Item Description STB']],left_on='Smartcard' ,right_on='Smart Card', how='left' )
ft_clone = merged


#### new sales
newsales_clone = newsales_clone.loc[newsales_clone['Customer Created Date'] == str((date-timedelta(days=1)))]
newsales_clone = newsales_clone.loc[~newsales_clone['Box Number'].isna()]
newsales_clone = newsales_clone.loc[~newsales_clone['Contract Status'].isna()]
newsales_clone = newsales_clone.loc[newsales_clone['Dealer Code'].str.lower().str.startswith('bein')]



# split data to (with_amount,no-amount)
newsales_clone_amount = newsales_clone.loc[~newsales_clone['Payment Amount'].isna()]
newsales_clone_no_amount = newsales_clone.loc[newsales_clone['Payment Amount'].isna()]

# link no amount with ft to get amount
merged = pd.merge(left=newsales_clone_no_amount, right=ft_clone[['Subscriber Nr','Created Date','Amount']], left_on=['Customer Number','Customer Created Date'], right_on=['Subscriber Nr','Created Date'], how='left')
merged = merged.drop(columns=['Subscriber Nr','Created Date','Payment Amount'])
merged = merged.rename(columns={'Amount':'Payment Amount'})

# concat with_amount with fixed no_amount
newsales_clone = pd.concat([merged,newsales_clone_amount])
newsales_clone= newsales_clone.drop_duplicates(keep='first')
newsales_clone['Sale Source'] = 'Dealers'
newsales_clone['Sale Type'] = 'New Sale'
newsales_clone.loc[newsales_clone['Dealer Type']=='Partner Head Office','Sale Source'] = 'Direct Sales'
newsales_clone["dataset"] = 'ft_clone'
newsales_clone['Location'] =''



# link with beindata to get SC,box type
merged = pd.merge(left=newsales_clone,right=beindata_clone[['Decoder','Smart Card','Item Description STB']], left_on='Box Number',right_on='Decoder',how='left')
merged.drop(columns=['Decoder'])
newsales_clone = merged

# link with FT to get MOP, FTNR, Bill Period

merged = pd.merge(left=newsales_clone,right=ft_clone[['Subscriber Nr','Created Date','Amount','Pay Mode','Ftnr','Bill Period']], left_on=['Customer Number','Customer Created Date','Payment Amount'], right_on=['Subscriber Nr','Created Date','Amount'],how='left')
merged.drop(columns=['Subscriber Nr','Created Date','Amount'])
merged.loc[merged['Pay Mode'].isna(),'Pay Mode'] = 'Cash'
newsales_clone = merged

newsales_clone['Start Date'] = pd.to_datetime(newsales_clone['Start Date'])
newsales_clone['End Date'] = pd.to_datetime(newsales_clone['End Date'])

newsales_clone['Contract Period'] = diff_month(newsales_clone['End Date'] , newsales_clone['Start Date'])




#### FT

# split into dealers , direct sales

ft_clone_dealers = ft_clone.loc[ft_clone['Default Entity Type'].isin(['Bein Dealer'])]
ft_clone_dealers = ft_clone_dealers.loc[ft_clone_dealers['Doc Type']=='JV']
ft_clone_dealers['Sale Source'] = 'Dealers'
ft_clone_dealers['Sale Type'] = 'Transaction'
ft_clone_dealers['dataset'] = 'ft_clone_dealers'

ft_clone_ds = ft_clone.loc[ft_clone['Default Entity Type'].isin(['Partner Head Office'])]
ft_clone_ds = ft_clone_ds.loc[ft_clone_ds['Doc Type']=='Payment']
ft_clone_ds['Sale Source'] = 'Direct Sales'
ft_clone_ds['Sale Type'] = 'Transaction'
ft_clone_ds['dataset'] = 'ft_clone_ds'



# ft_clone = ft_clone.loc[ft_clone['Default Entity Type'].isin(['Bein Dealer','Partner Head Office'])]



#Format the final results
# Transaction Date	LOCATION	AGENT	SERIAL	PACKAGE	PAYMENT PLAN	MOP	SALETYPE	BOX TYPE	TRANSACTION DATE	CSD	CED	AMOUNT	STAFF#	INVOICE	CONTRACT PERIOD	SALE TYPE
cols =  ['TRANSACTION DATE','LOCATION','AGENT','SERIAL','PAYMENT PLAN','MOP','SALE TYPE','BOX TYPE','CSD', 'CED','AMOUNT', 'FTNR','CONTRACT PERIOD','SALE SOURCE']



newsales_clone = newsales_clone[['Customer Created Date','Location','User Name','Smart Card','Plan','Pay Mode','Sale Type','Item Description STB','Start Date', 'End Date','Payment Amount', 'Ftnr','Contract Period','Sale Source']]
newsales_clone['Contract Period'] = diff_month(newsales_clone['End Date'] , newsales_clone['Start Date'])
newsales_clone.columns=cols
newsales_clone.to_csv('newsales.csv',index=False)

ft_final = pd.concat([ft_clone_dealers,ft_clone_ds])
ft_final['Start Date'] = ''
ft_final['End Date'] = ''
ft_final['Contract Period'] = ''



ft_final = ft_final[['Created Date','Location','User Name', 'Smartcard', 'Plan Name','Pay Mode','Sale Type','Item Description STB','Start Date', 'End Date', 'Amount','Ftnr','Contract Period','Sale Source']]
ft_final = ft_final.drop_duplicates()
ft_final.loc[ft_final['Pay Mode'].isna(),'Pay Mode'] = 'Cash'
ft_final.columns = cols



ft_final.to_csv('ft_final.csv',index=False)

all_final = pd.concat([ft_final,newsales_clone])
all_final = all_final.drop_duplicates()




all_final.to_csv("all_final.csv", index=False)


In [152]:
newsales.loc[newsales['Customer Number']=='19232125']



,Customer Number,Customer Created Date,Channel Provider,Customer Type,Box Number,Dealer Name,Dealer Code,Dealer Type,User Name,Customer Status,Balance,Start Date,End Date,Next Bill Date,Bill Frequency,Contract Status,Payment Amount,Plan,Ppv Balance,Account Balance,Total Balance
242,19232125,2026-01-20,beIN,Clubs,0381393708,CNE Head office,CNEHO,CNE Head Office,Moustafa Gamal Mohamed Abdel Latif,Active,14960 DR,20-01-2026,19-01-2027,20-01-2027,12M,Active,NaN,ComDTH- Premium,0 Dr,14960 Dr,14960 Dr


In [153]:
ft_clone.loc[ft_clone['Subscriber Nr']=='19232125']


,Subscriber Nr,Doc Type,Ftnr,Created Date,Created Time,Doc Status,Period From,Period To,Bank Date,Amount,User Name,User Fullname,Payment Ref No,Event Description,Payment Batch No,Batch Approval No,Default Entity Type,Collecting Entity,Pay Mode,Jv Type,Book Number,Smartcard,Bill Period,Bill Cycle,Invoice Type,Plan Name,Contract Number,Channel Provider,Subscriber Type,Subscriber Entity,Last Four Digits Of Card,Payment Flag,Location,Decoder,Smart Card,Item Description STB


In [154]:
newsales_clone

,TRANSACTION DATE,LOCATION,AGENT,SERIAL,PAYMENT PLAN,MOP,SALE TYPE,BOX TYPE,CSD,CED,AMOUNT,FTNR,CONTRACT PERIOD,SALE SOURCE
0,2026-01-21,,beIN Messdak agent4,10738784478,PREMIUM 09.24,Cash,New Sale,beIN 4k,2026-01-21,2027-01-20,1522,CNEJVN_586891,12,Dealers
1,2026-01-21,,beIN Alex Roshdy agent1,10738803971,ULTIMATE 09.24,Cash,New Sale,beIN 4k,2026-01-21,2027-01-20,5677,CNEJVN_586961,12,Dealers
2,2026-01-21,,beIN Tanta agent2,10738784577,PREMIUM 09.24,Cash,New Sale,beIN 4k,2026-01-21,2027-01-20,6087,CNEJVN_586900,12,Dealers
3,2026-01-21,,beIN Tanta agent4,10738786218,ULTIMATE 09.24,Cash,New Sale,beIN 4k,2026-01-21,2027-01-20,11354,CNEJVN_587012,12,Dealers
4,2026-01-21,,mohamed abdelmawla,10738805679,PREMIUM 09.24,Credit Card,New Sale,beIN 4k,2026-01-21,2027-01-20,6087,CNEPMT_2225717,12,Direct Sales
5,2026-01-21,,mohamed abdelmawla,10738805679,PREMIUM 09.24,Credit Card,New Sale,beIN 4k,2026-01-21,2027-01-20,6087,CNEPMT_2225717,12,Direct Sales
6,2026-01-21,,mohamed abdelmawla,10738805679,PREMIUM 09.24,Credit Card,New Sale,beIN 4k,2026-01-21,2027-01-20,6087,CNEPMT_2225717,12,Direct Sales
7,2026-01-21,,mohamed abdelmawla,10738805679,PREMIUM 09.24,Credit Card,New Sale,beIN 4k,2026-01-21,2027-01-20,6087,CNEPMT_2225717,12,Direct Sales
8,2026-01-21,,Maram Taher Youssef,10738916047,ULTIMATE 09.24,Credit Card,New Sale,beIN 4k,2026-01-21,2027-01-20,11354,CNEPMT_2225759,12,Direct Sales
9,2026-01-21,,mahmoud ibrahim,10738924553,PREMIUM 09.24,Credit Card,New Sale,beIN 4k,2026-01-21,2027-01-20,6087,CNEPMT_2225870,12,Direct Sales
